In [9]:
from sentence_transformers import SentenceTransformer
from transformers import pipeline as hf_pipeline
import faiss, numpy as np, json

print("Imports OK")

Imports OK


In [3]:
# Vérifier la version de transformers installée
import transformers
print(transformers.__version__)

4.40.0


In [17]:
import sys
print(sys.executable)

C:\Users\BrandlyFES\anaconda3\envs\DLxNLP\python.exe


In [14]:
# Modèle d'embedding : transforme texte → vecteur 384 dimensions
embedding_model = SentenceTransformer("multi-qa-MiniLM-L6-cos-v1")

# Modèle QA extractif : lit un contexte et extrait la réponse exacte
# C'est RoBERTa fine-tuné sur SQuAD 2.0 — jamais hallucine
qa_pipeline = hf_pipeline(
    "question-answering",
    model="deepset/roberta-base-squad2",
    framework="pt"
)

print("Modèles chargés !")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\BrandlyFES\anaconda3\envs\DLxNLP\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\BrandlyFES\.cache\huggingface\hub\models--sentence-transformers--multi-qa-MiniLM-L6-cos-v1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

C:\Users\BrandlyFES\anaconda3\envs\DLxNLP\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Modèles chargés !


In [12]:
# Charger la base de connaissances ENSA
with open("ensa_kb.json", "r", encoding="utf-8") as f:
    ensa_data = json.load(f)

# CORRECTION BUG #2 : reconstruire les passages dans le MÊME ordre que ensa_data
# Cela garantit que l'indice i dans FAISS = ensa_data[i]
passages_alignes = [
    item["question"] + " " + item["answer"]
    for item in ensa_data
]

# Charger l'index FAISS déjà construit
index = faiss.read_index("ensa_faiss.index")

print(f"KB chargée : {len(ensa_data)} entrées")
print(f"Index FAISS : {index.ntotal} vecteurs")
print(f"Alignement vérifié : {len(passages_alignes) == index.ntotal}")

KB chargée : 42 entrées
Index FAISS : 42 vecteurs
Alignement vérifié : True


In [15]:
def retrieve(query, top_k=3):
    """
    Étape 1 du pipeline RAG.
    
    - Encode la question en vecteur 384d
    - Cherche les top_k vecteurs les plus proches dans FAISS
    - Distance L2 : plus petit = plus proche = plus pertinent
    - Seuil : < 0.5 excellent, 0.5-1.0 bon, 1.0-1.2 limite, > 1.2 hors KB
    """
    # Transformer la question en vecteur
    query_vec = embedding_model.encode([query]).astype(np.float32)
    
    # Chercher dans l'index FAISS
    distances, indices = index.search(query_vec, top_k)
    
    # Construire les résultats — ALIGNEMENT GARANTI avec ensa_data
    results = []
    for i, idx in enumerate(indices[0]):
        if idx < len(ensa_data):  # sécurité
            results.append({
                "rank": i + 1,
                "score_faiss": float(distances[0][i]),
                "answer": ensa_data[idx]["answer"],
                "question": ensa_data[idx]["question"],
                "category": ensa_data[idx]["category"]
            })
    return results

# TEST IMMÉDIAT
print("=== TEST RETRIEVER ===")
r = retrieve("Comment s'inscrire à l'ENSA ?")
for p in r:
    print(f"  Rank {p['rank']} | score={p['score_faiss']:.3f} | {p['question'][:60]}")

=== TEST RETRIEVER ===
  Rank 1 | score=1.573 | Quels sont les enseignants du département Génie des Systèmes
  Rank 2 | score=1.596 | Quel est le contact de l'administration de l'ENSA Fès ?
  Rank 3 | score=1.641 | Quels sont les enseignants du département Mathématiques Appl


In [16]:
def read_answer(query, context):
    """
    Étape 2 du pipeline RAG.
    
    RoBERTa lit le contexte et prédit :
    - 'answer' : le span exact qui répond à la question
    - 'score'  : confiance du modèle (0.0 → 1.0)
    
    Différence critique avec GPT-2 :
    GPT-2  → génère du texte librement → peut inventer
    RoBERTa → extrait un span du contexte → ne peut pas inventer
    """
    result = qa_pipeline(
        question=query,
        context=context
    )
    return result["answer"], result["score"]

# TEST IMMÉDIAT
print("=== TEST READER ===")
contexte_test = "L'admission à l'ENSA Fès se fait via le concours national commun (CNC) pour les bacheliers scientifiques avec mention bien ou très bien en SM ou SP."
question_test = "Comment intégrer l'ENSA Fès ?"

reponse, score_qa = read_answer(question_test, contexte_test)
print(f"  Question : {question_test}")
print(f"  Réponse extraite : {reponse}")
print(f"  Score QA (confiance RoBERTa) : {score_qa:.3f}")

=== TEST READER ===
  Question : Comment intégrer l'ENSA Fès ?
  Réponse extraite : L'admission
  Score QA (confiance RoBERTa) : 0.010


In [17]:
SEUIL_FAISS = 1.2  # Distance L2 max acceptable avec all-MiniLM

def calculer_confiance(score_faiss, score_qa):
    """
    CORRECTION BUG #3 : formule combinée plus précise.
    
    score_faiss : distance L2 FAISS (0 = parfait, 1.2+ = hors KB)
    score_qa    : confiance RoBERTa (0.0 → 1.0)
    
    Formule :
      conf_faiss = 1 - (score_faiss / SEUIL)  → normalise FAISS entre 0 et 1
      confiance  = 60% FAISS + 40% QA         → combine les deux signaux
    
    Résultat : 0-100%
    Vert  > 70% : très fiable
    Orange 40-70% : probable
    Rouge  < 40% : incertain
    """
    conf_faiss = max(0.0, 1.0 - score_faiss / SEUIL_FAISS)
    confiance = round((0.6 * conf_faiss + 0.4 * score_qa) * 100)
    return confiance

# TEST
print("=== TEST SCORE CONFIANCE ===")
print(f"  FAISS=0.3, QA=0.9 → {calculer_confiance(0.3, 0.9)}%  (attendu: élevé)")
print(f"  FAISS=0.8, QA=0.6 → {calculer_confiance(0.8, 0.6)}%  (attendu: moyen)")
print(f"  FAISS=1.1, QA=0.2 → {calculer_confiance(1.1, 0.2)}%  (attendu: bas)")

=== TEST SCORE CONFIANCE ===
  FAISS=0.3, QA=0.9 → 81%  (attendu: élevé)
  FAISS=0.8, QA=0.6 → 44%  (attendu: moyen)
  FAISS=1.1, QA=0.2 → 13%  (attendu: bas)


In [25]:
# Installation si pas encore fait :
# pip install duckduckgo-search

from duckduckgo_search import DDGS

def fallback_web(query):
    """
    Déclenché quand score_faiss > SEUIL_FAISS (question hors KB).
    Cherche sur le web une réponse approximative.
    Confiance toujours basse (30%) pour signaler à l'utilisateur.
    """
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query + " ENSA Fès", max_results=3))
        if results:
            return {
                "reponse": results[0]["body"][:350],
                "source": "Web — " + results[0]["title"],
                "confiance": 30,
                "category": "web"
            }
    except Exception as e:
        print(f"  Fallback échoué : {e}")
    return None

In [26]:
def chatbot_ensa(query, top_k=3):
    """
    Pipeline RAG complet — version corrigée.
    
    Flux :
    1. Retriever  : FAISS trouve les top-3 passages pertinents
    2. Vérification : score > SEUIL → fallback web
    3. Reader     : RoBERTa extrait la réponse exacte du contexte
    4. Confiance  : formule combinée 60/40
    5. Retour     : réponse + confiance + source + catégorie
    """
    # ── ÉTAPE 1 : Retrieval ──────────────────────────────────
    passages = retrieve(query, top_k)
    if not passages:
        return {"reponse": "Erreur interne.", "confiance": 0}
    
    meilleur = passages[0]
    
    # ── ÉTAPE 2 : Vérification seuil ─────────────────────────
    if meilleur["score_faiss"] > SEUIL_FAISS:
        # Question hors base ENSA → fallback web
        fb = fallback_web(query)
        if fb:
            return fb
        return {
            "reponse": "Information non disponible dans ma base. Contactez contact@ensa-fes.ma",
            "confiance": 0,
            "source": None,
            "category": None
        }
    
    # ── ÉTAPE 3 : Extraction de la réponse (RoBERTa) ─────────
    # Combiner les top-3 passages pour plus de contexte
    contexte = " ".join([p["answer"] for p in passages])
    reponse, score_qa = read_answer(query, contexte)
    
    # ── ÉTAPE 4 : Score de confiance ─────────────────────────
    confiance = calculer_confiance(meilleur["score_faiss"], score_qa)
    
    return {
        "reponse": reponse,
        "source": meilleur["question"],
        "confiance": confiance,
        "category": meilleur["category"],
        "score_faiss": round(meilleur["score_faiss"], 3),
        "score_qa": round(score_qa, 3),
        "passage": meilleur["answer"]
    }